# نَبْض — توليد صوت المعلّمة للكتاب والملزمة
هذا الدفتر برنامج تشغيل لحزمة الأحياء الخاصة V7. **لا يحتوي كتاباً أو صوتاً خاصاً في GitHub**. بعد رفع الحزمة إلى حسابك في Colab، يُنتج مقاطع غير معتمدة للنشر إلى أن تُراجع مقابل PDF وتستمع إليها. استنساخ الصوت لا يضمن التطابق الكامل.


In [ ]:
!pip -q install 'git+https://github.com/resemble-ai/chatterbox.git' soundfile
import torch
print('GPU available:',torch.cuda.is_available())
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('جهاز التوليد:',DEVICE,'، المعالج العادي أبطأ من GPU')
from google.colab import drive,files
drive.mount('/content/drive')


In [ ]:
import json,zipfile,sys,os
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/Biology_Audio_V7')
ROOT.mkdir(parents=True,exist_ok=True)
if not (ROOT/'manifest.json').is_file():
  print('ارفع الملف الخاص Biology_Audio_App_v8.zip من هاتفك')
  uploaded=files.upload()
  packs=[x for x in uploaded if x.lower().endswith('.zip')]
  if not packs: raise RuntimeError('يجب اختيار ملف ZIP الخاص بالمشروع')
  with zipfile.ZipFile(packs[0]) as z:
    for entry in z.infolist():
      target=(ROOT/entry.filename).resolve()
      if not(target==ROOT.resolve() or ROOT.resolve() in target.parents): raise ValueError('ZIP غير آمن')
    z.extractall(ROOT)
sys.path.insert(0,str(ROOT/'tools'))
from generate_voice_batch import preflight
assert len(json.loads((ROOT/'manifest.json').read_text())['segments'])==1051
print(json.dumps(preflight(ROOT,allow_unverified=True,max_new=3),ensure_ascii=False,indent=2))


## تجربة صوت قصيرة
اعتمد الصوت بعد الاستماع بنفسك، لا تعتمد التشابه آلياً.


In [ ]:
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
from IPython.display import Audio,display
import torchaudio as ta
REF=ROOT/'audio/reference_voice_9s.wav'
display(Audio(filename=str(REF)))
model=ChatterboxMultilingualTTS.from_pretrained(device=DEVICE,t3_model='v3')
wav=model.generate('هسه نجي للجهاز العصبي. الخلية العصبية هي الوحدة الأساسية في بنائه.',language_id='ar',audio_prompt_path=str(REF),cfg_weight=0.3,exaggeration=0.5)
out=ROOT/'audio/voice_test_v8.wav'
ta.save(str(out),wav.cpu().unsqueeze(0) if wav.ndim==1 else wav.cpu(),model.sr)
display(Audio(filename=str(out)))
print('ملف التجربة:',out)


## توليد دفعة صوتية تجريبية
**غيّر VOICE_APPROVED إلى True فقط إذا اعتمدت الصوت**. النصوص المستخرجة غير مدققة؛ الأصوات الناتجة مسودات وتبقى خارج ملفات التطبيق المعتمدة.


In [ ]:
VOICE_APPROVED=False
if not VOICE_APPROVED: raise RuntimeError('استمع إلى العينة وأكّد قبول صوتها أولاً')
from generate_voice_batch import run
import gc
del model,wav
gc.collect()
torch.cuda.empty_cache()
print('generated:',run(ROOT,REF,max_new=10,source=None,start_page=1,allow_unverified=True,include_flagged=False,device=DEVICE))


## التدقيق والتصدير
شغّل المراجعة في الملفات الخاصة `review_text.html` و`review_audio.html`، ثم استورد ملفي قرارات المراجعة. قبل المراجعة سيحتوي التصدير على **صفر تسجيلات معتمدة**.


In [ ]:
from apply_review import apply
from export_app_assets import exported
for n in ('biology_text_review.json','biology_audio_review.json'):
 p=ROOT/n
 if p.exists(): print(n,apply(ROOT,p))
result=exported(ROOT,ROOT/'Biology_Approved_Audio_For_App_v8.zip',allow_demo=False)
print(json.dumps(result,ensure_ascii=False,indent=2))
print('بعد اعتماد الملفات: استورد Biology_Approved_Audio_For_App_v7.zip من زر تحديث الصوت في التطبيق')
